### Preparations

This notebook sets up basic Apache Iceberg infrastructure for the rest of the Iceberg notebooks and tests whether everything is up and running.

In [ ]:
lakekeeper_url = "http://lakekeeper:8181/"

minio_url  = "http://minio:9000"
minio_user = "minioadmin"
minio_pass = "minioadmin"

bucket_name = "warehouse"
warehouse_name = "iceberg"

In [ ]:
# create the 'warehouse' S3 bucket

import boto3


s3_resource = boto3.resource("s3", 
    endpoint_url=minio_url,
    aws_access_key_id=minio_user,
    aws_secret_access_key=minio_pass,
    aws_session_token=None,
    config=boto3.session.Config(signature_version='s3v4'),
    verify=False,
)
bucket = s3_resource.Bucket(bucket_name)

if (bucket.creation_date):
    print("Bucket already exists")
else:
    bucket.create()
    print("Bucket created")

In [ ]:
# bootstrap the catalog

import requests


r = requests.post(f"{lakekeeper_url}management/v1/bootstrap", json={"accept-terms-of-use": True})

try:
    r.raise_for_status()
    print("catalog bootsrapped")
except Exception as e:
    print(r.json().get("error"))

In [ ]:
# initialise the 'iceberg' warehouse

import requests


payload = {
  "warehouse-name": warehouse_name,
  "project-id": "00000000-0000-0000-0000-000000000000",
  "storage-profile": {
    "type": "s3",
    "bucket": bucket_name,
    "key-prefix": "iceberg",
    "assume-role-arn": None,
    "endpoint": minio_url,
    "region": "eu-central-1",
    "path-style-access": True,
    "flavor": "minio",
    "sts-enabled": True,
  },
  "storage-credential": {
    "type": "s3",
    "credential-type": "access-key",
    "aws-access-key-id": minio_user,
    "aws-secret-access-key": minio_pass,
  }
}

r = requests.post(f"{lakekeeper_url}management/v1/warehouse", json=payload)
r.json()

In [ ]:
# check the config of the created warehouse

import requests


r = requests.get(f"{lakekeeper_url}catalog/v1/config?warehouse={warehouse_name}")
r.json()

In [ ]:
# check that Spark client works

from pyspark.sql import SparkSession


spark = (
    SparkSession.builder
        .config(
            "spark.sql.extensions",
            "org.projectnessie.spark.extensions.NessieSparkSessionExtensions, org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
        )
        .config("spark.sql.catalog.iceberg", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.iceberg.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.iceberg.type", "rest")
        .config("spark.sql.catalog.iceberg.uri", "http://lakekeeper:8181/catalog/")
        .config("spark.sql.catalog.iceberg.warehouse", "iceberg")
        .config("spark.sql.catalog.iceberg.ref", "main")
        .config("spark.sql.catalog.iceberg.cache-enabled", False)
        .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')

spark.sql("""
    SHOW SCHEMAS
""").show()

In [ ]:
# check that Trino client works

from trino.dbapi import connect


trino_connection = connect(
    host="trino",
    port=8080,
    user="trino",
)
trino = trino_connection.cursor()
trino.execute("SHOW SCHEMAS FROM iceberg")

rows = trino.fetchall()
print(rows)